# 006 第一次可重跑的研究练习

先读配置，再看主运行与全部描述运行。不要事后改变主种子。

In [1]:
from pathlib import Path
import json
from experiment import prepare, run, digest, mae
from test_experiment import run_tests, verify_artifacts
base = Path.cwd()
config = json.loads((base / "config.json").read_text())
print(config)

{'protocol_version': 1, 'primary_seed': 101, 'descriptive_seeds': [101, 102, 103], 'train_sample_count': 8, 'candidate_biases': [0, 1, 2], 'fixed_coefficient': 2, 'metric': 'mean_absolute_error', 'model_tie_break': 'affine', 'split_by': 'device', 'data_files': {'train': 'train.csv', 'validation': 'validation.csv', 'test_inputs': 'test_inputs.csv', 'test_labels': 'test_labels.csv'}}


## 在测试标签之前冻结
prepare不读取测试标签。检查实际入选编号、候选分数和验证选择。

In [2]:
frozen, trials, inputs, label_path = prepare()
for trial in trials:
    print("seed", trial["seed"], "IDs", trial["selected_train_ids"])
    print("train", trial["train_mae"], "validation", trial["validation_mae"])
print("selected", frozen["selected_model"])
print("predictions", frozen["test_predictions"])

seed 101 IDs ['B3', 'D3', 'C1', 'B2', 'B4', 'A1', 'D2', 'A4']
train {'constant_mean': 2.25, 'affine': 0.25} validation {'constant_mean': 2.0, 'affine': 0.0}
seed 102 IDs ['B1', 'C3', 'C2', 'B2', 'A3', 'C1', 'D2', 'C4']
train {'constant_mean': 1.875, 'affine': 0.625} validation {'constant_mean': 2.0, 'affine': 0.0}
seed 103 IDs ['D3', 'D4', 'C4', 'D2', 'D1', 'A4', 'A2', 'C2']
train {'constant_mean': 1.75, 'affine': 0.5} validation {'constant_mean': 2.0, 'affine': 0.0}
selected {'name': 'affine', 'coefficient': 2, 'bias': 1}
predictions [3.0, 5.0, 7.0, 9.0]


## 独立手算主运行
预期八项误差和2、平均0.25，均值基线为6。

In [3]:
actual = [7, 7, 3, 5, 10, 3, 4, 9]
predicted = [7, 7, 3, 5, 9, 3, 5, 9]
print("MAE", mae(actual, predicted))
print("constant", sum(actual) / len(actual))
assert mae(actual, predicted) == 0.25
assert sum(actual) / len(actual) == 6

MAE 0.25
constant 6.0


## 保存完整运行并核验摘要
run每次创建独立编号目录，先保存冻结计划，再揭示公开教学标签；只有run_status为complete才是完整结果。

In [4]:
output = base / "notebook_outputs"
result = run(output)
print(result)
print("artifact check", verify_artifacts(output))
print("config SHA-256", digest(base / "config.json"))
assert result["test_mae"] == 2

{'selected_model': {'name': 'affine', 'coefficient': 2, 'bias': 1}, 'primary_seed': 101, 'test_mae': 2.0, 'test_sample_count': 4, 'test_device_count': 1, 'scope': 'Public synthetic descriptive exercise; not a blind or population-level finding'}
artifact check True
config SHA-256 c1766014a8becc0734fe4ae1fb2cfc3ee69ab8d8ec2baae48ffa99117bee24f6


## 全套故障与重跑检查
临时删除标签、改标签、改设备、改样本数和改结果字节；另检验先成功、后失败的两次尝试不会混合产物。不修改原始数据。

In [5]:
checks = run_tests()
print(checks)
assert checks["status"] == "passed"

{'status': 'passed', 'test_groups': ['exact selected IDs, candidate scores and hand-derived parameters', 'all declared descriptive seeds and final hand-derived score', 'two complete reruns produce identical artifacts', 'one-byte artifact change invalidates recorded digest', 'test labels physically absent during fitting and model selection', 'changing test labels does not affect frozen choices or predictions', 'entity overlap rejected', 'impossible sample-count configuration rejected', 'success then changed-plan failure in same collection preserves old run and marks new failure', 'fresh process from unrelated directory reproduces final result', 'empty, mismatched and nonfinite metric inputs rejected']}


## 不同预测为何验证同分
按四条标签展开误差，而不只看平均。

In [6]:
labels = [3, 5, 7, 9]
for constant in [6, 5.375, 6.5]:
    errors = [abs(constant - y) for y in labels]
    print(constant, errors, sum(errors) / len(errors))
    assert sum(errors) / len(errors) == 2

6 [3, 1, 1, 3] 2.0
5.375 [2.375, 0.375, 1.625, 3.625] 2.0
6.5 [3.5, 1.5, 0.5, 2.5] 2.0


## 写出三条待解释现象
1. 训练子集变化而偏移量不变。
2. 不同常数预测验证同分。
3. 验证零误差，测试四行一致低估2。

每条配具体记录、解释候选和下一步证据；不作统计或因果断言。最后重启并全部执行。